In [0]:
data = [
    (1, "Ravi", "Hyderabad", 50000),
    (2, "Suresh", "Bangalore", 65000),
    (3, "Anil", "Hyderabad", 55000),
    (4, "Kiran", "Chennai", 45000),
    (5, "Ravi", "Hyderabad", 50000),
    (6, "Mahesh", None, 70000)
    (7, "balu","goa",90000)
]

columns = ["id", "name", "city", "salary"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_customer")

In [0]:
%sql
SELECT * FROM bronze_customer;

In [0]:
bronze_df = spark.table("bronze_customer")

display(bronze_df)


In [0]:
silver_df = bronze_df.dropDuplicates(["id"])

In [0]:
from pyspark.sql.functions import coalesce, lit

silver_df = silver_df.withColumn(
    "city",
    coalesce("city", lit("Unknown"))
)

In [0]:
display(silver_df)


In [0]:
silver_df = bronze_df.dropDuplicates(["name", "city", "salary"])

In [0]:
silver_df = silver_df.withColumn(
    "city",
    coalesce("city", lit("Unknown"))
)


In [0]:
display(silver_df)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customer")

In [0]:
%sql
SELECT * FROM silver_customer;

In [0]:
silver_df = spark.table("silver_customer")

In [0]:
from pyspark.sql.functions import count, avg, round

gold_df = silver_df.groupBy("city").agg(
    count("id").alias("employee_count"),
    round(avg("salary"), 2).alias("average_salary")
)

In [0]:
display(gold_df)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_city_summary")

In [0]:
%sql
SELECT * 
FROM gold_customer_city_summary;